In [7]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [8]:
import re

from jitgen.executors.python import InProcPythonExecutor
from jitgen.prebuilt.python import create_python_async_jitgen_session

from langsmith import traceable

from langchain_core.language_models import BaseChatModel

SYSTEM_PROMPT = """You are a helpful assistant assigned with the task of problem-solving. To achieve this,
you will be using an interactive coding environment equipped with a variety of tool
functions to assist you throughout the process.
At each turn, you should first provide your step-by-step thinking for solving the task.
Your thought process should be enclosed using "<thought>" tag, for example: <thought>
I need to print "Hello World!" </thought>.
After that, you have two options:
1) Interact with a Python programming environment and receive the corresponding output.
Your code should be enclosed using "<execute>" tag, for example: <execute> print("
Hello World!") </execute>.
2) Directly provide a solution that adheres to the required format for the given task.
Your solution should be enclosed using "<solution>" tag, for example: The answer is <
solution> A </solution>.
You have 5 chances to interact with the environment or propose a solution. You can only
propose a solution 2 times.
"""

@traceable
async def stream_async_execution(llm: BaseChatModel, prompt: str):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    response = ""
    observation = "Observation:\n"
    async with create_python_async_jitgen_session(
        start_marker="<execute>", end_marker="</execute>"
    ) as session:

        @session.on_stdout
        def on_stdout(stdout: str):
            nonlocal observation
            observation += stdout

        async for chunk in llm.astream(messages):
            chunk_text = chunk.text
            response += chunk_text
            await session.apush(chunk_text)
            yield chunk_text

    messages.append({"role": "assistant", "content": response})
    messages.append({"role": "human", "content": observation})
    yield "\n" + observation

    async for chunk in llm.astream(messages):
        chunk_text = chunk.text
        yield chunk_text


async def _execute_python_code(text: str) -> str:
    code_block = re.search(r"<execute>(.+)", text, re.DOTALL)
    if not code_block:
        return ""

    source_code = code_block.group(1)

    executor = InProcPythonExecutor()
    result = await executor.aexecute(source_code)

    if result.success:
        return result.output
    else:
        raise ValueError(f"Error detected. Halting further processing. {result.error}")

@traceable
async def stream_sync_execution(llm: BaseChatModel, prompt: str):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    response = ""
    async for chunk in llm.astream(messages):
        chunk_text = chunk.text
        response += chunk_text
        yield chunk_text

    observation = "Observation:\n" + await _execute_python_code(response)

    messages.append({"role": "assistant", "content": response})
    messages.append({"role": "human", "content": observation})
    yield "\n" + observation

    async for chunk in llm.astream(messages):
        chunk_text = chunk.text
        yield chunk_text

In [9]:
from random import randint
import time

session_id = randint(0, 1000000)
def init_model() -> BaseChatModel:
    return ChatOllama(
        model="xingyaow/codeact-agent-mistral",
        temperature=0,
        seed=session_id,
        cache=False,
        keep_alive=0
    )

problem = "Provide a sum of squares of all prime numbers less than or equal to 30."

print("=== Stream async execution ===")
start_time = time.perf_counter()
async for chunk in stream_async_execution(init_model(), problem):
    if chunk.startswith("\nObservation:"):
        print(f"First observation received {time.perf_counter() - start_time:.2f} seconds", end="")
print(f"\n\nTime taken: {time.perf_counter() - start_time:.2f} seconds")

print("\n\n=== Stream sync execution ===")
start_time = time.perf_counter()
async for chunk in stream_sync_execution(init_model(), problem):
    if chunk.startswith("\nObservation:"):
        print(f"First observation received {time.perf_counter() - start_time:.2f} seconds", end="")
print(f"\n\nTime taken: {time.perf_counter() - start_time:.2f} seconds")


=== Stream async execution ===
First observation received 4.30 seconds

Time taken: 6.98 seconds


=== Stream sync execution ===
First observation received 4.32 seconds

Time taken: 6.92 seconds


In [14]:
from random import randint
import time
import matplotlib.pyplot as plt

N = 5  # default number of runs


async def _measure_execution(run_gen):
    start = time.perf_counter()
    first_observation_time = None

    async for chunk in run_gen:
        if first_observation_time is None and chunk.startswith("\nObservation:"):
            first_observation_time = time.perf_counter() - start

    total_time = time.perf_counter() - start
    return first_observation_time, total_time


def init_model(session_id) -> BaseChatModel:
    return ChatOllama(
        # model="xingyaow/codeact-agent-mistral",
        model="nemotron-cascade-2:latest",
        temperature=0,
        seed=session_id,
    )

async def reset_model(session_id):
    await ChatOllama(
        # model="xingyaow/codeact-agent-mistral",
        model="nemotron-cascade-2:latest",
        temperature=0,
        seed=session_id,
        cache=False,
        keep_alive=0,
    ).ainvoke("Hi")

async def benchmark_executions(n: int = N, prompt: str = "Provide a sum of squares of all prime numbers less than or equal to 150."):
    async_first, async_total = [], []
    sync_first, sync_total = [], []

    session_id = randint(0, 1000000)
    for _ in range(n):
        await reset_model(session_id)
        first_obs, total = await _measure_execution(stream_async_execution(init_model(session_id), prompt))
        async_first.append(first_obs)
        async_total.append(total)
        await reset_model(session_id)
        first_obs, total = await _measure_execution(stream_sync_execution(init_model(session_id), prompt))
        sync_first.append(first_obs)
        sync_total.append(total)

    runs = list(range(1, n + 1))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

    axes[0].plot(runs, async_first, marker="o", label="async")
    axes[0].plot(runs, sync_first, marker="o", label="sync")
    axes[0].set_title("Time to First Observation")
    axes[0].set_xlabel("Run")
    axes[0].set_ylabel("Seconds")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(runs, async_total, marker="o", label="async")
    axes[1].plot(runs, sync_total, marker="o", label="sync")
    axes[1].set_title("Total Execution Time")
    axes[1].set_xlabel("Run")
    axes[1].set_ylabel("Seconds")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    return {
        "async": {"first_observation": async_first, "total": async_total},
        "sync": {"first_observation": sync_first, "total": sync_total},
    }


benchmark_results = await benchmark_executions()

CancelledError: 